In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Membaca file CSV
file_path = '/content/drive/MyDrive/PBA DATA/data_berita.csv'
df = pd.read_csv(file_path)
df

,Judul,Berita
0,"Dapat Nomor Urut 3, Ganjar: Persatuan Indonesia","Menurut dia, semangat persatuan Indonesia itu..."
1,"Prabowo: Kalau Pemilu Curang, Mengkhianati Ban...","JAKARTA, KOMPAS.com - Calon presiden (capres) ..."
2,Nestle Indonesia PHK 126 Karyawan di Pabrik Ja...,"JAKARTA, KOMPAS.com - PT Nestle Indonesia mela..."
3,Lirik dan Chord Lagu I Tried to Leave You - Le...,Lagu bergenre pop tersebut dirilis pada 1988 m...
4,"Lirik Lagu Another Life, Lagu Baru dari PinkPa...",Di lagu ini ia berkolaborasi dengan penyanyi a...
...,...,...
95,Catherine Wilson dan Idham Masse Batal Cerai,Permohonan cerai yang dilayangkan oleh Idham M...
96,Mutasi TNI: Laksda Budi Purwanto Jadi Danpushi...,Penunjukkan itu berdasarkan Surat Keputusan Pa...
97,"Seorang Perempuan Tipu Caleg DPRD DKI, Modusny...","Kapolsek Tambora Kompol Putra Pratama berujar,..."
98,"Hamil Anak Kedua, Cesen Eks JKT48: Lebih Agak ...","Cesen mengatakan, kehamilan kedua ini sangat b..."


In [ ]:
df['Closeness Centrality'] = ""
df['PageRank'] = ""
df['Eigen Vector'] = ""
def ringkasan(news):
    berita = df["Berita"].iloc[news]

    # Tahap 1: Menghapus karakter non-alphanumeric kecuali spasi dan titik
    text_no_special_chars = re.sub(r'[^\w\s.]', '', berita)

    # Tahap 2: Mengubah teks menjadi huruf kecil (case folding)
    text_lowercase = text_no_special_chars.lower()

    # Tahap 3: Menghapus stopwords
    stop_words = set(stopwords.words('indonesian'))
    stop_words.discard('tidak')
    words = text_lowercase.split()
    filtered_words = [word for word in words if word.lower() not in stop_words]
    join_text = ' '.join(filtered_words)

    # Tahap 4: Tokenisasi Kalimat
    kalimat = re.split(r'(?<=\.) +', join_text)

    if len(kalimat) < 2:
        # Jika hanya ada satu kalimat setelah tokenisasi, lewati perhitungan cosine similarity dan centrality
        df.at[news, 'Closeness Centrality'] = "Tidak cukup data untuk ringkasan"
        df.at[news, 'PageRank'] = "Tidak cukup data untuk ringkasan"
        df.at[news, 'Eigen Vector'] = "Tidak cukup data untuk ringkasan"
        return

    # TF-IDF
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform(kalimat)

    # Cosine Similarity
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

    # Graph
    G = nx.Graph()
    # Tambahkan node (kalimat)
    for i in range(len(cosine_sim)):
        G.add_node(i)

    # Tambahkan edge (hubungan) antara kalimat berdasarkan cosine similarity
    for i in range(len(cosine_sim)):
        for j in range(i + 1, len(cosine_sim)):  # Hindari pengulangan dan self-loop
            similarity = cosine_sim[i][j]
            if similarity > 0.2:  # Ambang batas untuk menentukan hubungan
                G.add_edge(i, j, weight=similarity)

    # Closeness Centrality
    closeness_centrality = nx.closeness_centrality(G)
    pagerank = nx.pagerank(G)
    try:
        eigenvector = nx.eigenvector_centrality(G, max_iter=1000, tol=1e-06)
    except nx.PowerIterationFailedConvergence:
        eigenvector = {}

    # Membuat DataFrame dari nilai Closeness Centrality
    centrality_df = pd.DataFrame(closeness_centrality.items(), columns=['Node', 'Closeness Centrality'])
    pagerank_df = pd.DataFrame(pagerank.items(), columns=['Node', 'PageRank'])
    eigenvector_df = pd.DataFrame(eigenvector.items(), columns=['Node', 'Eigen Vector'])

    # Mengurutkan DataFrame berdasarkan Closeness Centrality dari yang terbesar
    centrality_df_sorted = centrality_df.sort_values(by='Closeness Centrality', ascending=False)
    pagerank_df_sorted = pagerank_df.sort_values(by='PageRank', ascending=False)
    eigenvector_df_sorted = eigenvector_df.sort_values(by='Eigen Vector', ascending=False)

    # Mengambil tiga baris teratas dari DataFrame yang sudah diurutkan
    top_three_closeness = centrality_df_sorted.head(3)
    top_three_pagerank = pagerank_df_sorted.head(3)
    top_three_eigenvector = eigenvector_df_sorted.head(3)

    # Mendapatkan indeks node dari tiga baris teratas
    top_node_closeness = top_three_closeness['Node'].tolist()
    top_node_pagerank = top_three_pagerank['Node'].tolist()
    top_node_eigenvector = top_three_eigenvector['Node'].tolist()

    # Menyimpan kalimat yang sesuai dengan node-node teratas ke kolom DataFrame
    summary_closeness = ' '.join([kalimat[node] for node in top_node_closeness])
    summary_pagerank = ' '.join([kalimat[node] for node in top_node_pagerank])
    summary_eigenvector = ' '.join([kalimat[node] for node in top_node_eigenvector])

    df.at[news, 'Closeness Centrality'] = summary_closeness
    df.at[news, 'PageRank'] = summary_pagerank
    df.at[news, 'Eigen Vector'] = summary_eigenvector

In [ ]:
for index in range(len(df)):
    ringkasan(index)

In [ ]:
output_file_path = '/content/drive/MyDrive/PBA DATA/data_berita_ringkasan.csv'
df.to_csv(output_file_path, index=False)